In [2]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

In [17]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [19]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class EvaluationScoreSchema(BaseModel):
    feedback: str = Field(description="Feedback on improving the essay")
    score: float = Field(description="Score of the essay from 0 to 10", ge=0, le=10)
    

In [20]:
llm_structured  = llm.with_structured_output(EvaluationScoreSchema)

In [21]:
#State Definition
import operator
from typing import Annotated


class EssayEvaluationState(TypedDict):
    essay: str
    language_feedback: str
    clarity_feedback: str
    factual_accuracy_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[float], operator.add, Field(description="List of individual scores for each criterion")]
    avg_score: float
    

In [22]:
from langchain_core.prompts import PromptTemplate

def evaluate_language(EssayEvaluationState: EssayEvaluationState) :
    prompt_template = PromptTemplate(
        template = "Act as a language expert and evaluate the language, grammar, fluency of the following essay and provide feedback and a score from 0 to 10:\n\n{essay}",
        input_variables = ["essay"],
        
    )
    chain = prompt_template | llm_structured
    language_evaluation = chain.invoke({"essay": EssayEvaluationState["essay"]})
    updated_state = { "language_feedback": language_evaluation['feedback'], "individual_scores": [language_evaluation['score']] } #type: ignore
    
    return updated_state

def evaluate_clarity(EssayEvaluationState: EssayEvaluationState) :
    prompt_template = PromptTemplate(
        template = "Act as a clarity expert and evaluate the clarity, coherence, and structure of the following essay and provide feedback and a score from 0 to 10:\n\n{essay}",
        input_variables = ["essay"],
        
    )
    chain = prompt_template | llm_structured
    clarity_evaluation = chain.invoke({"essay": EssayEvaluationState["essay"]})
    updated_state = { "clarity_feedback": clarity_evaluation['feedback'], "individual_scores": EssayEvaluationState["individual_scores"] + [clarity_evaluation['score']] } #type: ignore
    
    return updated_state

def evaluate_factual_accuracy(EssayEvaluationState: EssayEvaluationState) :
    prompt_template = PromptTemplate(
        template = "Act as a factual accuracy expert and evaluate the correctness of the mentioned facts and claims in the following essay and provide feedback and a score from 0 to 10:\n\n{essay}",
        input_variables = ["essay"],
        
    )
    chain = prompt_template | llm_structured
    factual_accuracy_evaluation = chain.invoke({"essay": EssayEvaluationState["essay"]})
    updated_state = { "factual_accuracy_feedback": factual_accuracy_evaluation['feedback'], "individual_scores": EssayEvaluationState["individual_scores"] + [factual_accuracy_evaluation['score']] } #type: ignore
    
    return updated_state

In [23]:
graph = StateGraph(EssayEvaluationState)

#add nodes
graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_clarity", evaluate_clarity)
graph.add_node("evaluate_factual_accuracy", evaluate_factual_accuracy)

In [24]:
from langchain_core.output_parsers import StrOutputParser

def calculate_overall_feedback(EssayEvaluationState: EssayEvaluationState) :
    avg_score = sum(EssayEvaluationState["individual_scores"]) / len(EssayEvaluationState["individual_scores"])
    
    prompt_template = PromptTemplate(
        template = "Based on the following feedback and scores, provide an overall feedback for the essay:\n\nLanguage Feedback: {language_feedback}\nClarity Feedback: {clarity_feedback}\nFactual Accuracy Feedback: {factual_accuracy_feedback}\nAverage Score: {avg_score}\n\nINSTRUCTIONS: If the average score is below 5, provide constructive criticism and suggestions for improvement. If the average score is between 5 and 7, provide feedback on areas of improvement and highlight strengths. If the average score is above 7, provide positive feedback and highlight strengths.",
        input_variables = ["language_feedback", "clarity_feedback", "factual_accuracy_feedback", "avg_score"],
        
    )
    
    parser = StrOutputParser()
    chain = prompt_template | llm | parser
    
    overall_feedback = chain.invoke({
        "language_feedback": EssayEvaluationState['language_feedback'],
        "clarity_feedback": EssayEvaluationState['clarity_feedback'],
        "factual_accuracy_feedback": EssayEvaluationState['factual_accuracy_feedback'],
        "avg_score": avg_score
    })
    
    updated_state = {
        "overall_feedback": overall_feedback,
        "avg_score": avg_score
    }
    
    return updated_state

In [25]:
graph.add_node("calculate_overall_feedback", calculate_overall_feedback)

In [26]:
#add edges
graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_clarity")
graph.add_edge(START, "evaluate_factual_accuracy")
graph.add_edge("evaluate_language",  "calculate_overall_feedback")
graph.add_edge("evaluate_clarity",  "calculate_overall_feedback")
graph.add_edge("evaluate_factual_accuracy",  "calculate_overall_feedback")
graph.add_edge("calculate_overall_feedback", END)

In [27]:
workflow = graph.compile()

In [28]:
sample_essay = """In the modern era, technology has become an integral part of our daily lives. From smartphones to artificial intelligence, technological advancements have transformed the way we communicate, work, and entertain ourselves. However, while technology offers numerous benefits, it also presents challenges that society must address. One of the most significant concerns is the impact of technology on privacy. With the rise of social media and data collection practices, individuals' personal information is often at risk of being exploited. Additionally, the rapid pace of technological change can lead to job displacement, as automation and AI systems replace human labor in various industries. Therefore, it is crucial for policymakers and stakeholders to implement regulations that protect privacy and ensure a fair transition for workers affected by technological disruptions."""

In [29]:
input_state = {
    "essay" : sample_essay
}

In [30]:
final_state = workflow.invoke(input_state)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


TypeError: 'EvaluationScoreSchema' object is not subscriptable